# Session 4: Convolutional Neural Networks and Filters

This notebook contains structured tasks based on Session 4 materials.

**Instructions:**
- Do NOT use a separate `.py` file. Complete all tasks directly in this notebook.
- Keep your code clear and well-commented.
- For Visulaization, please use `%matplotlib inline`
- Submit your completed notebook by uploading it to your forked repository.

### ***DO NOT UPLOAD THE DATASET TO GITHUB as it's too large***
<hr>

## Task 1: Convolutional Filters on an Image
Apply several 2D filters (Sobel X, Sobel Y, Laplacian, and Sharpen) to an image and visualize the results side by side.

**Hint:**
- Use OpenCV (`cv2.filter2D`) or NumPy to apply kernels.
- Define common kernels (Sobel, Laplacian, Sharpen).
- Use `matplotlib.pyplot` to plot the original and filtered images.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
# --- Load an image ---
# Read in color (BGR) and convert to RGB for correct matplotlib display
img = cv2.imread('./Snail.jpeg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# --- Convert to grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- Define kernels ---
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)

sobel_y = np.array([[-1, -2, -1],
                    [0,  0,  0],
                    [1,  2,  1]], dtype=np.float32)

laplacian = np.array([[0,  1, 0],
                      [1, -4, 1],
                      [0,  1, 0]], dtype=np.float32)

sharpen = np.array([[0, -1,  0],
                    [-1, 5, -1],
                    [0, -1,  0]], dtype=np.float32)

# --- Apply filters using cv2.filter2D ---
sobel_x_img = cv2.filter2D(gray, -1, sobel_x)
sobel_y_img = cv2.filter2D(gray, -1, sobel_y)
laplacian_img = cv2.filter2D(gray, -1, laplacian)
sharpen_img = cv2.filter2D(gray, -1, sharpen)

# --- Plot results ---
titles = ["Original (RGB)", "Grayscale", "Sobel X", "Sobel Y", "Laplacian", "Sharpen"]
images = [img_rgb, gray, sobel_x_img, sobel_y_img, laplacian_img, sharpen_img]

plt.figure(figsize=(12, 8))
for i in range(len(images)):
    plt.subplot(2, 3, i+1)
    cmap = 'gray' if len(images[i].shape) == 2 else None
    plt.imshow(images[i], cmap=cmap)
    plt.title(titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()


## Task 2: Build a Simple CNN on FashionMNIST
Construct and train a small Convolutional Neural Network (CNN) on the FashionMNIST dataset.

**Hint:**
- Use `torchvision.datasets.FashionMNIST` for loading data.
- Define a small CNN with `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, and `nn.Linear`.
- Train for a few epochs using an optimizer (e.g., Adam) and loss function (e.g., CrossEntropyLoss).
- Print training accuracy after each epoch.

In [ ]:
# --- TODO: Import libraries ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# --- TODO: Load FashionMNIST dataset with transforms ---
transform = transforms.Compose([
    transforms.ToTensor(),                # Convert PIL -> Tensor
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# --- TODO: Define a simple CNN class ---
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3)  # 1 input channel (grayscale), 8 output filters
        self.fc1 = nn.Linear(8*26*26, 10)            # 26x26 after conv, 10 classes

    def forward(self, x):
        x = F.relu(self.conv1(x))      # Apply conv + ReLU
        x = x.view(x.size(0), -1)      # Flatten
        x = self.fc1(x)                # Fully connected
        return x

# Instantiate model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- TODO: Training loop (2-3 epochs) ---
for epoch in range(3):  # run 3 epochs
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward + optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/3], Loss: {running_loss/len(train_loader):.4f}")

print("Training finished ✅")


## Task 3: Visualize Feature Maps
Visualize feature maps produced by the first convolutional layer of your CNN.

**Hint:**
- Take a single image from the test set.
- Pass it through the first convolutional layer (`net.conv1`).
- Convert the output to NumPy and plot several channels as images.

In [ ]:
images, labels = next(iter(test_loader))
image = images[0].unsqueeze(0)  # Take first image, keep batch dimension

# --- Forward pass through the first conv layer ---
model.eval()
with torch.no_grad():
    feature_maps = model.conv1(image)   # shape: [1, num_filters, H, W]

# --- Convert to numpy ---
feature_maps = feature_maps.squeeze(0).cpu().numpy()  # remove batch dimension

# --- Plot the first few feature maps in a grid ---
num_maps = min(6, feature_maps.shape[0])  # show up to 6 maps
fig, axes = plt.subplots(1, num_maps, figsize=(15, 5))

for i in range(num_maps):
    axes[i].imshow(feature_maps[i], cmap='gray')
    axes[i].axis("off")
    axes[i].set_title(f"Map {i+1}")

plt.show()


## Task 4: Haar Cascade Face Detection

Apply Haar Cascade classifiers for basic object detection (faces).

**Hint:**
- Use `cv2.CascadeClassifier` with a pre-trained XML file (e.g., `haarcascade_frontalface_default.xml`).
- Convert the input image to grayscale before detection.
- Use `detectMultiScale` to get bounding boxes.
- Draw rectangles on detected faces with `cv2.rectangle`.
- Visualize with Matplotlib.


In [ ]:
import cv2
import matplotlib.pyplot as plt

# --- Load an image containing a face ---
img = cv2.imread("./face1.jpeg")   # <-- replace with your image path
if img is None:
    raise ValueError("Image not found! Check the path.")

# --- Convert to grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- Load Haar Cascade XML (frontal face) ---
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# --- Detect faces ---
faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30, 30)
)

# --- Draw bounding boxes on the original image ---
for (x, y, w, h) in faces:
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

# --- Convert BGR to RGB for matplotlib display ---
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# --- Display results ---
plt.imshow(img_rgb)
plt.axis("off")
plt.title("Detected Faces")
plt.show()
